# Five-model training, validation, and independent testing

Run the selected settings for RF, KNN, DT, MLP, and SVR using consistent evaluation methods. Choose a model with `method` and export its result workbook. Hyperparameter selection is handled separately in `FiveModels_Tuning.ipynb`.

In [ ]:
import pandas as pd
import numpy as np
import time

from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.feature_selection import f_regression
from sklearn.svm import SVR
from sklearn.multioutput import MultiOutputRegressor
from sklearn.compose import TransformedTargetRegressor
import sklearn


In [ ]:
from pathlib import Path

# Prefer the repository dataset; allow the original Google Colab location when available.
local_candidates = [
    Path('data/PALMO_NACA_4Series_Complete_Dataset.xlsx'),
    Path('PALMO_NACA_4Series_Complete_Dataset.xlsx'),
    Path('PALMO_NACA_4Series_Complete_Dataset-2.xlsx'),
]
input_file = next((path for path in local_candidates if path.exists()), None)
if input_file is None:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        input_file = Path('/content/drive/MyDrive/Test_data/PALMO_NACA_4Series_Complete_Dataset.xlsx')
    except ImportError:
        raise FileNotFoundError('Place the PALMO Excel file in data/ or the current directory.')

df = pd.read_excel(input_file, sheet_name='PALMO_4Series_Data')
x = df[['%Camber', '%Thickness', 'Reynolds#', 'Mach#', 'Alpha']]
y = df[['cl', 'cd', 'cm']]
print('Dataset:', input_file)
print('Features used:', list(x.columns))


In [ ]:
range_df = pd.DataFrame({
    'Variable': list(x.columns) + list(y.columns),
    'Type': ['Input'] * len(x.columns) + ['Output'] * len(y.columns),
    'Minimum': list(x.min()) + list(y.min()),
    'Maximum': list(x.max()) + list(y.max()),
    'Range': list(x.max() - x.min()) + list(y.max() - y.min())
})

range_df

In [ ]:
method = 'RF'  # RF, KNN, DT, MLP, or SVR

method = method.upper()

if method not in ['RF', 'KNN', 'MLP', 'DT', 'SVR']:
    raise ValueError("Choose 'RF', 'KNN', 'MLP', 'DT', or 'SVR'.")

test_airfoils = pd.DataFrame({
    'Airfoil': ['NACA 3415', 'NACA 3418', 'NACA 4415', 'NACA 4421'],
    '%Camber': [3, 3, 4, 4],
    '%Thickness': [15, 18, 15, 21]
})

test_mask = (
    ((df['%Camber'] == 3) & (df['%Thickness'].isin([15, 18]))) |
    ((df['%Camber'] == 4) & (df['%Thickness'].isin([15, 21])))
)

x_base = x.loc[~test_mask].copy()
y_base = y.loc[~test_mask].copy()
x_test = x.loc[test_mask].copy()
y_test = y.loc[test_mask].copy()

base_airfoils = (
    df.loc[~test_mask, ['%Camber', '%Thickness']]
    .drop_duplicates()
    .sort_values(['%Camber', '%Thickness'])
    .reset_index(drop=True)
)

data_split_df = pd.DataFrame({
    'Group': ['12 base airfoils', '4 unseen test airfoils', 'Complete dataset'],
    'Rows': [len(x_base), len(x_test), len(df)]
})

print('Base airfoil combinations:')
print(base_airfoils)
print('\nUnseen test airfoils:')
print(test_airfoils)
print('\nData split:')
print(data_split_df)
print('\nSelected model:', method)

if len(base_airfoils) != 12 or len(x_test) == 0:
    raise ValueError('The 12/4 airfoil split was not created correctly.')

In [ ]:
output_results = {
    'cl': {'R2': [], 'MAE': [], 'RMSE': []},
    'cd': {'R2': [], 'MAE': [], 'RMSE': []},
    'cm': {'R2': [], 'MAE': [], 'RMSE': []}
}

In [ ]:
def create_model(method, random_state):
    if method == 'RF':
        return RandomForestRegressor(
            n_estimators=300,
            max_depth=20,
            min_samples_leaf=1,
            min_samples_split=2,
            max_features=1.0,
            random_state=random_state,
            n_jobs=-1
        )

    if method == 'DT':
        return DecisionTreeRegressor(
            max_depth=20,
            min_samples_leaf=1,
            min_samples_split=2,
            random_state=random_state
        )

    if method == 'KNN':
        return KNeighborsRegressor(
            n_neighbors=3,
            weights='distance',
            p=1,
            n_jobs=-1
        )

    if method == 'SVR':
        return TransformedTargetRegressor(
            regressor=MultiOutputRegressor(
                SVR(
                    kernel='rbf',
                    C=10.0,
                    epsilon=0.05,
                    gamma=0.2,
                    cache_size=256,
                    tol=0.001,
                    max_iter=-1
                ),
                n_jobs=1
            ),
            transformer=StandardScaler()
        )

    if method == 'MLP':
        return MLPRegressor(
            hidden_layer_sizes=(128, 64),
            alpha=0.0001,
            learning_rate_init=0.003,
            max_iter=500,
            random_state=random_state
        )

    raise ValueError('Unknown model: ' + str(method))


train_r2_list = []
train_mae_list = []
train_rmse_list = []

val_r2_list = []
val_mae_list = []
val_rmse_list = []

run_times = []

for i in range(10):
    print(f'Performing training for model {i+1}')

    X_train_df, X_val_df, y_train, y_val = train_test_split(
        x_base, y_base, train_size=0.7, random_state=i
    )

    if method in ['KNN', 'MLP', 'SVR']:
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train_df)
        X_val = scaler.transform(X_val_df)
    else:
        X_train = X_train_df
        X_val = X_val_df

    clf = create_model(method, i)

    start = time.time()
    clf.fit(X_train, y_train.values)
    y_pred_train = clf.predict(X_train)
    y_pred_val = clf.predict(X_val)
    end = time.time()

    run_times.append(end - start)

    train_r2_list.append(r2_score(y_train.values, y_pred_train))
    train_mae_list.append(mean_absolute_error(y_train.values, y_pred_train))
    train_rmse_list.append(np.sqrt(mean_squared_error(y_train.values, y_pred_train)))

    val_r2_list.append(r2_score(y_val.values, y_pred_val))
    val_mae_list.append(mean_absolute_error(y_val.values, y_pred_val))
    val_rmse_list.append(np.sqrt(mean_squared_error(y_val.values, y_pred_val)))

    for j, target in enumerate(['cl', 'cd', 'cm']):
        output_results[target]['R2'].append(
            r2_score(y_val[target].values, y_pred_val[:, j])
        )
        output_results[target]['MAE'].append(
            mean_absolute_error(y_val[target].values, y_pred_val[:, j])
        )
        output_results[target]['RMSE'].append(
            np.sqrt(mean_squared_error(y_val[target].values, y_pred_val[:, j]))
        )

final_model = create_model(method, 0)

if method in ['KNN', 'MLP', 'SVR']:
    final_scaler = StandardScaler()
    X_base_final = final_scaler.fit_transform(x_base)
    X_test_final = final_scaler.transform(x_test)
else:
    X_base_final = x_base
    X_test_final = x_test

final_start = time.time()
final_model.fit(X_base_final, y_base.values)
y_pred_base = final_model.predict(X_base_final)
y_pred_test = final_model.predict(X_test_final)
final_runtime = time.time() - final_start

In [ ]:
per_output_rows = []

for target in ['cl', 'cd', 'cm']:
    per_output_rows.append({
        'Output': target,
        'Validation R2': np.mean(output_results[target]['R2']),
        'R2 Std': np.std(output_results[target]['R2']),
        'Validation MAE': np.mean(output_results[target]['MAE']),
        'MAE Std': np.std(output_results[target]['MAE']),
        'Validation RMSE': np.mean(output_results[target]['RMSE']),
        'RMSE Std': np.std(output_results[target]['RMSE'])
    })

per_output_df = pd.DataFrame(per_output_rows)
per_output_df

In [ ]:
test_rows = []

for j, target in enumerate(['cl', 'cd', 'cm']):
    test_rows.append({
        'Output': target,
        'Test R2': r2_score(y_test[target].values, y_pred_test[:, j]),
        'Test MAE': mean_absolute_error(y_test[target].values, y_pred_test[:, j]),
        'Test RMSE': np.sqrt(mean_squared_error(y_test[target].values, y_pred_test[:, j]))
    })

test_results_df = pd.DataFrame(test_rows)
test_results_df


In [ ]:
train_r2_avg = np.mean(train_r2_list)
train_mae_avg = np.mean(train_mae_list)
train_rmse_avg = np.mean(train_rmse_list)

val_r2_avg = np.mean(val_r2_list)
val_mae_avg = np.mean(val_mae_list)
val_rmse_avg = np.mean(val_rmse_list)

val_r2_std = np.std(val_r2_list)
val_mae_std = np.std(val_mae_list)
val_rmse_std = np.std(val_rmse_list)

test_r2 = r2_score(y_test.values, y_pred_test)
test_mae = mean_absolute_error(y_test.values, y_pred_test)
test_rmse = np.sqrt(mean_squared_error(y_test.values, y_pred_test))

print('Average training R2:', round(train_r2_avg, 4))
print('Fixed-setting validation R2:', round(val_r2_avg, 4), '+/-', round(val_r2_std, 4))
print('Final unseen-airfoil test R2:', round(test_r2, 4))
print('Final unseen-airfoil test MAE:', round(test_mae, 4))
print('Final unseen-airfoil test RMSE:', round(test_rmse, 4))

In [ ]:
F_rows = []

for target in ['cl', 'cd', 'cm']:
    F_vals, p_vals = f_regression(x_base, y_base[target])

    for feature, F_value, p_value in zip(x_base.columns, F_vals, p_vals):
        F_rows.append([
            target,
            feature,
            F_value,
            p_value
        ])

f_test_df = pd.DataFrame(F_rows, columns=[
    'Output',
    'Feature',
    'F_statistic',
    'p_value'
])

f_test_df

In [ ]:
validation_iterations_df = pd.DataFrame({
    'Iteration': list(range(1, 11)),
    'Validation R2': val_r2_list,
    'Validation MAE': val_mae_list,
    'Validation RMSE': val_rmse_list,
    'Runtime': run_times
})

summary_rows = pd.DataFrame({
    'Iteration': ['Average', '+/-'],
    'Validation R2': [val_r2_avg, val_r2_std],
    'Validation MAE': [val_mae_avg, val_mae_std],
    'Validation RMSE': [val_rmse_avg, val_rmse_std],
    'Runtime': [np.mean(run_times), np.std(run_times)]
})

validation_iterations_df = pd.concat(
    [validation_iterations_df, summary_rows],
    ignore_index=True
)

results_df = pd.DataFrame({
    'Metric': [
        'Average Training R2',
        'Average Training MAE',
        'Average Training RMSE',
        'Validation R2',
        'Validation MAE',
        'Validation RMSE',
        'Unseen Test R2',
        'Unseen Test MAE',
        'Unseen Test RMSE',
        'Final Model Runtime'
    ],
    method: [
        train_r2_avg,
        train_mae_avg,
        train_rmse_avg,
        str(round(val_r2_avg, 4)) + ' +/- ' + str(round(val_r2_std, 4)),
        str(round(val_mae_avg, 4)) + ' +/- ' + str(round(val_mae_std, 4)),
        str(round(val_rmse_avg, 4)) + ' +/- ' + str(round(val_rmse_std, 4)),
        test_r2,
        test_mae,
        test_rmse,
        str(round(final_runtime, 2)) + ' s'
    ]
})

results_df
results_df.loc[len(results_df)] = ['Validation procedure', 'Fixed settings; diagnostic splits, not nested tuning validation']

results_df.loc[len(results_df)] = ['Model description', method]
results_df.loc[len(results_df)] = ['scikit-learn version', sklearn.__version__]
results_df.loc[len(results_df)] = ['Selected settings', str(create_model(method, 0).get_params(deep=True))]


In [ ]:
definitions = pd.DataFrame({
    'Metric': [
        'R2',
        'MAE',
        'RMSE',
        'Validation',
        'Unseen Test',
        'F_statistic',
        'p_value'
    ],
    'Definition': [
        'R2 measures how well the model explains the variation in the output values',
        'Mean Absolute Error: average absolute difference between predicted and actual values',
        'Root Mean Square Error: square root of the average squared prediction error',
        'Diagnostic 70/30 row splits of the 12 base airfoils with fixed selected settings. Settings were previously selected on all base rows, so these splits are not independent of tuning. Label these results as fixed-settings diagnostics; do not mix them with independent nested-validation results.',
        'Final results for NACA 3415, 3418, 4415, and 4421, which were never used for training or validation',
        'Measures how strongly each input feature is related to the output',
        'Under the null of zero linear association, the probability of an F statistic at least as large as observed'
    ]
})

definitions

In [ ]:
output_file = method + '_12Base_4Unseen_FixedSettings_results.xlsx'

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    results_df.to_excel(
        writer,
        sheet_name='Model_Results',
        index=False
    )

    per_output_df.to_excel(
        writer,
        sheet_name='Per_Output_Validation',
        index=False
    )

    validation_iterations_df.to_excel(
        writer,
        sheet_name='Validation_Iterations',
        index=False
    )

    test_results_df.to_excel(
        writer,
        sheet_name='Unseen_Test_Results',
        index=False
    )

    test_airfoils.to_excel(
        writer,
        sheet_name='Held_Out_Airfoils',
        index=False
    )

    data_split_df.to_excel(
        writer,
        sheet_name='Data_Split',
        index=False
    )

    range_df.to_excel(
        writer,
        sheet_name='Data_Ranges',
        index=False
    )

    f_test_df.to_excel(
        writer,
        sheet_name='F_Test',
        index=False
    )

    definitions.to_excel(
        writer,
        sheet_name='Definitions',
        index=False
    )

print('Saved results to:', output_file)